### 실습 문제

함수 호출(Function Calling)로 **간단한 환율 변환기**를 만들어 보세요.

1. 더미 함수 `convert_krw(amount, currency)` 를 정의하세요. 고정 환율로 원화(KRW) 금액을 외화로 변환해 반환합니다. (예: USD=1350, JPY=9, EUR=1450 원)
2. 이 함수를 `tools` 스키마로 등록하세요.
3. `"30000원은 몇 달러야?"` 같은 질문으로 모델을 호출하고, 모델이 요청한 함수를 실제로 실행한 뒤, 그 결과를 다시 모델에 전달하여 **최종 자연어 답변**을 출력하세요. (이메일/SMTP는 사용하지 않습니다.)

### 테스트 입력 예시

- `"30000원은 몇 달러야?"` → `convert_krw(amount=30000, currency="USD")` 호출 → 약 22.2 달러 안내

## 정답

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
import json

client = OpenAI()
Model = "gpt-5-nano"

In [2]:
# 1. 더미 환율 변환 함수 (고정 환율)
def convert_krw(amount, currency):
    """원화(KRW) 금액을 지정한 통화로 변환한다."""
    rates = {"USD": 1350, "JPY": 9, "EUR": 1450}  # 1 단위당 원화 가격
    currency = currency.upper()
    if currency not in rates:
        return json.dumps({"error": f"지원하지 않는 통화: {currency}"})
    converted = amount / rates[currency]
    return json.dumps(
        {"amount_krw": amount, "currency": currency, "converted": round(converted, 2)}
    )


# 2. tools 스키마 정의
tools = [{
    "type": "function",
    "name": "convert_krw",
    "description": "원화(KRW) 금액을 지정한 통화(USD, JPY, EUR)로 변환한다.",
    "parameters": {
        "type": "object",
        "properties": {
            "amount": {"type": "number", "description": "변환할 원화 금액"},
            "currency": {"type": "string", "description": "목표 통화 코드 (USD, JPY, EUR)"},
        },
        "required": ["amount", "currency"],
    },
}]

# 3-1. 모델 호출 (함수 호출 유도)
messages = [{"role": "user", "content": "30000원은 몇 달러야?"}]
response = client.responses.create(model=Model, input=messages, tools=tools)

# 3-2. 모델 출력(추론 + 함수 호출)을 대화에 추가하고, 함수를 실제 실행
for item in response.output:
    messages.append(item)
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = convert_krw(**args)
        messages.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": str(result),
        })

# 3-3. 함수 결과를 포함해 다시 호출 -> 최종 자연어 답변
final = client.responses.create(model=Model, input=messages, tools=tools)
print(final.output_text)

대략 22.22 USD입니다. (30,000원 ≈ 22.22달러)

참고로 환율은 시시각각 변동되니, 정확한 금액은 거래 시점의 환율을 확인해 주세요. 원하시면 다른 통화로도 계산해 드릴게요.
